# What Is Hypothesis Testing

In [61]:
import pandas as pd
from trial import get_columns_by_type
from preprocessing import percentage_to_int, join_and_sort
from pathlib import Path
import shutil
from category_encoders import OneHotEncoder, TargetEncoder
from joblib import load, dump
import pandas as pd
import numpy as np
from sklearn import set_config
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import ElasticNet, PoissonRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import make_scorer, mean_absolute_error
from transformers import DateTransformer
from scipy.stats import f
set_config(transform_output="pandas")

## Read Dataset

In [63]:
df = pd.read_csv('../data/preprocessed.csv', index_col="fixture_id")
df.drop("Unnamed: 0", axis=1, inplace=True)
df['start_time'] = pd.to_datetime(df['start_time'])
df.drop("passing_accuracy", axis=1, inplace=True)
col_types = get_columns_by_type(df)
df['start_time'].max()

## TRAIN TEST SPLIT

In [67]:
X = df.drop(['home_goals', 'away_goals'], axis=1)
home_goals = df['home_goals'].to_numpy()
away_goals = df['away_goals'].to_numpy()
X_train, X_test, y_home_train, y_home_test, y_away_train, y_away_test = train_test_split(X, home_goals, away_goals, test_size=0.25, shuffle=False)

## Create Pipeline

In [73]:
# custom pipeline
mae_scorer = make_scorer(mean_absolute_error)
date_transformer = Pipeline(steps=[
        ('transformer', DateTransformer()),
        ('encoder', TargetEncoder())])
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])
feature_preprocessor = ColumnTransformer(transformers=[
    ('numerical', numeric_transformer, col_types["numeric"]),
    ('datetime', date_transformer, col_types["timestamp"])
])

In [74]:
# define estimators
linear_params = {
    'estimator__alpha': [0.03, 0.05, 0.1],
    'estimator__l1_ratio': [0.1, 0.3, 0.5, 0.7]
}
linear = Pipeline(steps=[
    ('preprocessor', feature_preprocessor),
    ('estimator', ElasticNet())
])

In [75]:
home_model = GridSearchCV(linear, param_grid=linear_params)
away_model = GridSearchCV(linear, param_grid=linear_params)

## Training

In [76]:
home_model.fit(X_train, y_home_train)

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning:

Skipping features without any observed values: ['away_min_rating' 'rating' 'home_max_rating' 'away_std_rating'
 'home_min_rating' 'home_std_rating' 'home_mean_rating' 'away_max_rating'
 'away_mean_rating']. At least one non-missing value is needed for imputation with strategy='median'.

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning:

Skipping features without any observed values: ['away_min_rating' 'rating' 'home_max_rating' 'away_std_rating'
 'home_min_rating' 'home_std_rating' 'home_mean_rating' 'away_max_rating'
 'away_mean_rating']. At least one non-missing value is needed for imputation with strategy='median'.

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning:

Skipping features without any observed values: ['away_min_rating' '

GridSearchCV(estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('numerical',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['H_GA_OTR',
                                                                          'away_max_duels_won_percentage',
                                                                          'away_top_assistor_9',
                                                                          'away_max_key_passes',
                                                                          'home_mean_blocks',
                                                                          'away_max_passing_accuracy',
                                                                          'home_std_fouls_committe...
                                                                          'home_max_key_passes',
                                                                          'home_min_duels_won_percentage',
                                                                          'home_top_assistor_3',
                                                                          'home_max_rating',
                                                                          'home_top_scorer_5',
                                                                          'away_min_blocks', ...]),
                                                                        ('datetime',
                                                                         Pipeline(steps=[('transformer',
                                                                                          DateTransformer()),
                                                                                         ('encoder',
                                                                                          TargetEncoder())]),
                                                                         ['start_time'])])),
                                       ('estimator', ElasticNet())]),
             param_grid={'estimator__alpha': [0.03, 0.05, 0.1],
                         'estimator__l1_ratio': [0.1, 0.3, 0.5, 0.7]})

In [77]:
away_model.fit(X_train, y_away_train)

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning:

Skipping features without any observed values: ['away_min_rating' 'rating' 'home_max_rating' 'away_std_rating'
 'home_min_rating' 'home_std_rating' 'home_mean_rating' 'away_max_rating'
 'away_mean_rating']. At least one non-missing value is needed for imputation with strategy='median'.

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning:

Skipping features without any observed values: ['away_min_rating' 'rating' 'home_max_rating' 'away_std_rating'
 'home_min_rating' 'home_std_rating' 'home_mean_rating' 'away_max_rating'
 'away_mean_rating']. At least one non-missing value is needed for imputation with strategy='median'.

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning:

Skipping features without any observed values: ['away_min_rating' '

GridSearchCV(estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('numerical',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['H_GA_OTR',
                                                                          'away_max_duels_won_percentage',
                                                                          'away_top_assistor_9',
                                                                          'away_max_key_passes',
                                                                          'home_mean_blocks',
                                                                          'away_max_passing_accuracy',
                                                                          'home_std_fouls_committe...
                                                                          'home_max_key_passes',
                                                                          'home_min_duels_won_percentage',
                                                                          'home_top_assistor_3',
                                                                          'home_max_rating',
                                                                          'home_top_scorer_5',
                                                                          'away_min_blocks', ...]),
                                                                        ('datetime',
                                                                         Pipeline(steps=[('transformer',
                                                                                          DateTransformer()),
                                                                                         ('encoder',
                                                                                          TargetEncoder())]),
                                                                         ['start_time'])])),
                                       ('estimator', ElasticNet())]),
             param_grid={'estimator__alpha': [0.03, 0.05, 0.1],
                         'estimator__l1_ratio': [0.1, 0.3, 0.5, 0.7]})

## Evaluation

In [78]:
# training predictions
home_train_pred = home_model.predict(X_train)
away_train_pred = away_model.predict(X_train)

# test predictions
home_test_pred = home_model.predict(X_test)
away_test_pred = away_model.predict(X_test)

# evaluate TODO add adjusted r2
home_train_mae = mean_absolute_error(home_train_pred, y_home_train) 
home_test_mae = mean_absolute_error(home_test_pred, y_home_test)

away_train_mae = mean_absolute_error(away_train_pred, y_away_train)
away_test_mae = mean_absolute_error(away_test_pred, y_away_test)

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning:

Skipping features without any observed values: ['away_min_rating' 'rating' 'home_max_rating' 'away_std_rating'
 'home_min_rating' 'home_std_rating' 'home_mean_rating' 'away_max_rating'
 'away_mean_rating']. At least one non-missing value is needed for imputation with strategy='median'.

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning:

Skipping features without any observed values: ['away_min_rating' 'rating' 'home_max_rating' 'away_std_rating'
 'home_min_rating' 'home_std_rating' 'home_mean_rating' 'away_max_rating'
 'away_mean_rating']. At least one non-missing value is needed for imputation with strategy='median'.

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning:

Skipping features without any observed values: ['away_min_rating' '

In [79]:
print(away_train_mae, away_test_mae)
print(home_train_mae, home_test_mae)

0.8682718448897434 0.9226433678528674
0.9021312135761665 1.0211273690671516


In [53]:
# I am trying to present my findings from a data science project to stakeholders. I am doing a prediction of the number of goals scored by the home team and a separate model to predict the number of goals scored by the away team. The stakeholders want to know. Is it harder to predict home team score or away team score? Here are my findings:

# I have a training set of 2500 samples. 
# The standard deviation of the home team score (y_hat_1) is 1.301.
# The standard deviation of the away team score (y_hat_2) is 1.199.
# My hunch is that given there is a larger variance in the home team

In [54]:
# F - Statistic Calculation

In [80]:
def get_f_stat(s1, s2):
    s1_dof = s1.size - 1
    s2_dof = s2.size - 1

    if s1.var() > s2.var():
        return s1.var() / s2.var(), s1_dof, s2_dof
    return s2.var() / s1.var(), s2_dof, s1_dof


In [97]:
f_stat, dfn, dfd = get_f_stat(y_away_train, y_home_train)

In [92]:
fig = px.histogram(x=y_away_train, histnorm='percent')
fig.update_layout(bargap=0.2, xaxis={"title": "away_goals_scored"})
fig.show()

In [96]:
fig = px.histogram(x=y_home_train, histnorm='percent')
fig.update_layout(bargap=0.2, xaxis={"title": "home_goals_scored"})
fig.show()

In [106]:
import plotly.graph_objects as go

In [121]:
# cummulative dstribution functions side by side comparison
fig = go.Figure()
fig.add_trace(go.Histogram(x=y_home_train, name="home team", histnorm='percent'))
fig.add_trace(go.Histogram(x=y_away_train, name="away team", histnorm='percent'))
fig.update_layout(
    title="Premier League Home Field Advantage Histogram", 
    xaxis={"title": "Number of Goals Scored"},
    yaxis={"title": "Percent"},
    bargap=0.2)
fig.show()

# Residual Plots

In [58]:
import plotly.express as px

In [59]:
y_home_test, home_test_pred

(array([2, 2, 5, 1, 3, 3, 3, 1, 0, 2, 1, 2, 5, 2, 0, 2, 2, 0, 1, 0, 4, 5,
        1, 2, 2, 0, 1, 1, 1, 1, 0, 3, 0, 4, 0, 0, 1, 0, 3, 0, 3, 1, 0, 0,
        1, 3, 0, 3, 2, 1, 0, 0, 0, 1, 2, 1, 2, 3, 0, 3, 1, 1, 0, 2, 1, 3,
        0, 2, 1, 2, 2, 0, 2, 4, 0, 1, 2, 0, 0, 2, 2, 3, 7, 1, 2, 2, 1, 1,
        1, 1, 0, 0, 2, 0, 0, 3, 0, 0, 1, 1, 2, 1, 0, 1, 2, 1, 1, 1, 1, 0,
        3, 0, 3, 2, 4, 3, 2, 1, 4, 3, 2, 2, 0, 4, 1, 0, 2, 1, 4, 1, 1, 1,
        1, 2, 0, 1, 1, 1, 2, 3, 3, 1, 1, 0, 1, 3, 1, 2, 2, 2, 2, 1, 1, 3,
        3, 0, 0, 4, 3, 0, 7, 2, 0, 2, 1, 3, 1, 0, 0, 2, 3, 6, 2, 0, 1, 2,
        1, 1, 1, 3, 1, 1, 0, 3, 1, 0, 2, 2, 3, 2, 2, 0, 4, 2, 1, 1, 3, 1,
        2, 2, 2, 3, 1, 2, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 2, 0, 1, 3, 1, 2,
        1, 2, 3, 2, 0, 1, 0, 0, 3, 0, 1, 0, 0, 2, 2, 1, 3, 0, 2, 0, 0, 2,
        2, 2, 2, 1, 1, 6, 2, 2, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 2, 0, 4, 0,
        1, 2, 4, 5, 1, 4, 1, 0, 0, 2, 3, 2, 0, 1, 2, 1, 2, 0, 0, 0, 1, 2,
        0, 2, 3, 2, 0, 0, 1, 1, 2, 1, 

In [60]:
home_residuals = (y_home_test - home_test_pred)
fig = px.scatter(x=home_test_pred, y=home_residuals)
fig.show()

# Z-Test Hypothesis Testing

Let's start with home residuals. Our MAE is 0.95, which means we're off by nearly a goal on average.

In [18]:
home_residuals = np.abs((y_home_test - home_test_pred))
away_residuals = np.abs((y_away_test - away_test_pred))

In order to get an accurate hypothesis test, we need to be pulling from a normal distribution. Let's check to see if this plot is normally distributed

In [20]:
home_residuals.size

761

In [21]:
from statsmodels.stats.diagnostic import lilliefors

In [22]:
def is_noramlly_distributed(sample):
    statistic, pvalue = lilliefors(sample)
    if pvalue > 0.05:
        return True
    return False

In [23]:
sample_normal = np.random.normal(loc=0, scale=1, size=6000)

In [24]:
fig = px.histogram(x=sample_normal)
fig.update_layout(bargap=0.1)

In [25]:
is_noramlly_distributed(sample_normal)

True

In [26]:
is_noramlly_distributed(home_residuals)

False

# Bootstrapping

In [128]:
from mlxtend.evaluate import bootstrap

In [130]:
bootstrap(np.abs(home_residuals), np.mean, 761)

(1.0218114735044155,
 0.03264396797030118,
 (0.955687158055175, 1.0841503849055614))

In [28]:
# normal resids
statistic, pvalue = lilliefors(normal_residuals)
print(f"PVALUE: {pvalue}")
if pvalue > 0.05:
    print("NORMALLY DISTRIBUTED")
else:
    print("NOT NORMALLY DISTRIBUTED")

NameError: name 'normal_residuals' is not defined

In [ ]:
sum(normal_residuals) / 1000

0.9940065681444984

In [ ]:
fig = px.histogram(x=normal_residuals)
fig.update_layout(bargap=0.1)

In [ ]:
def get_test_statistic():
    pooled_std = np.append(y_home_test, y_away_test).std()
    pooled_std
    t_statistic = (home_test_mae - away_test_mae) / (pooled_std * np.sqrt((1 / y_home_test.size) + (1 / y_away_test.size)))
    return t_statistic

In [ ]:
# residual plot

In [ ]:
(y_away_test - away_test_pred).std()

1.183980859281045

# Save Experiment

In [ ]:
trial_number = 0

In [ ]:
# store model and feature importances with joblib
Path.mkdir(f'../data/trial_{trial_number}')
Path.mkdir(f'../data/trial_{trial_number}/models')
Path.mkdir(f'../data/trial_{trial_number}/features')

In [ ]:
def get_feature_importances(pipeline, X_test):
    regressor = pipeline['regressor'].best_estimator_
    coefs = list(np.round(regressor.coef_, 3))
    features = pipeline['preprocessor'].transform(X_test).columns.to_list()
    zipped = list(zip(coefs, features))
    feature_importances = sorted(zipped, key=lambda x: abs(x[0]), reverse=True)
    return feature_importances

In [ ]:
dump(home_pipeline, f'../data/trial_{trial_number}/models/home_pipeline.joblib')
dump(away_pipeline, f'../data/trial_{trial_number}/models/away_pipeline.joblib')

home_features = get_feature_importances(home_pipeline, X_test)
away_features = get_feature_importances(away_pipeline, X_test)

dump(home_features, f'../data/trial_{trial_number}/features/home_features.joblib')
dump(away_features, f'../data/trial_{trial_number}/features/away_features.joblib')

# create hyperparameter table
trial_results = pd.DataFrame({
    'home_train_mae': [home_train_mae],
    'home_test_mae': [home_test_mae],
    'away_train_mae': [away_train_mae],
    'away_test_mae': [away_test_mae],
    'dataset_path': [f'../data/trial_{trial_number}/prem_data.csv'],
    'home_model_path': [f'../data/trial_{trial_number}/home_pipeline.joblib'],
    'away_model_path': [f'../data/trial_{trial_number}/away_pipeline.joblib'],
    'test_size': [0.1],
})

# write to disk
home_ht = Path(f'../data/trial_{trial_number}/hyperparameter_table.csv')
if home_ht.exists():
    ht = pd.concat([pd.read_csv(home_ht), trial_results])
    ht.to_csv(f'../data/hyperparameter_table.csv')
else:
    trial_results.to_csv(f'../data/trial_{trial_number}/hyperparameter_table.csv')

Index(['start_time', 'start_time_weekday', 'start_time_is_weekend'], dtype='object')
Index(['start_time', 'start_time_weekday', 'start_time_is_weekend'], dtype='object')


In [ ]:
from mlxtend.evaluate import bias_variance_decomp

In [ ]:
set_config(transform_output="numpy")

In [ ]:
X_train.to_numpy()

array([[0, nan, 73.0, ..., 0, 0, 0],
       [1, nan, 88.0, ..., 0, 0, 0],
       [2, nan, 95.0, ..., 0, 0, 0],
       ...,
       [2734, 8.5, 40.0, ..., 0, 2, 2],
       [2735, 6.7, 3.0, ..., 3, 5, 0],
       [2736, 6.3, 3.0, ..., 6, 5, 3]], dtype=object)

In [ ]:
# preprocess the data
preprocessor = home_pipeline["preprocessor"]
numpy_x_train = preprocessor.fit_transform(X_train).to_numpy()
numpy_x_test = preprocessor.fit_transform(X_test).to_numpy()

Index(['start_time', 'start_time_weekday', 'start_time_is_weekend'], dtype='object')
Index(['start_time', 'start_time_weekday', 'start_time_is_weekend'], dtype='object')


In [ ]:
home_estimator = home_pipeline["regressor"].best_estimator_
away_estimator = away_pipeline["regressor"].best_estimator_
# pass to bvd

In [ ]:
# Bias Variance Tradeoff
mse, bias, var = bias_variance_decomp(home_estimator, numpy_x_train, y_home_train, numpy_x_test, y_home_test, loss='mse', num_rounds=200, random_seed=2)
# summarize results
print('MSE: %.3f' % mse)
print('Bias: %.3f' % bias)
print('Variance: %.3f' % var)

NameError: name 'bias_variance_decomp' is not defined

In [ ]:
# Bias Variance Tradeoff
mse, bias, var = bias_variance_decomp(away_estimator, numpy_x_train, y_away_train, numpy_x_test, y_away_test, loss='mse', num_rounds=200, random_seed=1)
# summarize results
print('MSE: %.3f' % mse)
print('Bias: %.3f' % bias)
print('Variance: %.3f' % var)

MSE: 1.350
Bias: 1.343
Variance: 0.006


In [ ]:
# Interpret Results of ML Extend

np.sqrt(1.349)

1.161464592658769